# Particle Physics Lab — NumPy to Higgs

Welcome! This is the same lab as `demo-before-cadence.ipynb`, but wired up with **Cadence**. As you work, your progress is sent to a teacher dashboard — they can see in real time how the class is doing, which checkpoints are tripping people up, and which solutions are the fastest.

All the cells still run normally on your machine. The only difference: a few `check(...)` calls have been added.


## Setup

Two lines that bring you into the class — `%load_ext cadence` enables the magics, and `%cadence_session` joins you under your name. The teacher provided the `join_code`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

%load_ext cadence

# ⚠️ Replace <JOIN_CODE> with the one your teacher gave you (printed when they ran
# demo-teacher-setup.ipynb, or any %cadence_create_lesson cell). Replace
# "Your name here" with your actual name so the dashboard can attribute attempts.
%cadence_session <JOIN_CODE> "Your name here"

from cadence import check, mark_done


## Part 1 — NumPy warmup


### Exercise 1: array statistics

Create an array containing the integers `0, 1, 2, …, 99` and compute its mean.

*Hint:* `np.arange` returns a half-open range.


In [ ]:
arr = np.arange(100)
mean_value = arr.mean()

check("setup.mean-value", mean_value)


### Exercise 2: row sums

Given the 3×4 matrix below, return the sum of each row as a list of three numbers.

*Hint:* `sum` takes an `axis=` keyword.


In [ ]:
M = np.arange(12).reshape(3, 4)
print(M)

row_sums = M.sum(axis=1).tolist()

check("setup.row-sums", row_sums)


### Exercise 3: boolean masking

A simulated detector reports 1 000 particle energies in GeV (drawn from an exponential distribution with mean 50 GeV). **How many of them exceed 100 GeV?**

Return your answer as a plain Python `int`.


In [ ]:
rng = np.random.default_rng(42)
energies = rng.exponential(scale=50.0, size=1000)

n_above_100 = int((energies > 100).sum())

check("setup.n-above-100", n_above_100)


## Part 2 — Particle four-vectors

In high-energy physics we describe each particle by a *four-vector* `(E, px, py, pz)`. Collider experiments typically report `(pT, η, φ, m)` instead, where:

- `pT` is the transverse momentum (in the plane perpendicular to the beam),
- `η` is the pseudorapidity (angle along the beam, but log-stretched),
- `φ` is the azimuthal angle around the beam axis,
- `m` is the rest mass (often 0 for photons, ~0.105 GeV for muons).

The conversion is:

$$
p_x = p_T \cos\varphi, \qquad p_y = p_T \sin\varphi, \qquad p_z = p_T \sinh\eta
$$
$$
E = \sqrt{p_x^2 + p_y^2 + p_z^2 + m^2}
$$


### Exercise 4: invariant mass of a photon pair

The invariant mass of two particles with four-vectors $p_1, p_2$ is

$$
m_{\gamma\gamma}^2 \;=\; (E_1 + E_2)^2 \;-\; \bigl(\vec p_1 + \vec p_2\bigr)^2
$$

Two photons land in the calorimeter:

- γ₁: `pT = 60 GeV`, `η =  1.2`, `φ = 0.3`
- γ₂: `pT = 70 GeV`, `η = -0.4`, `φ = 2.5`

Write `inv_mass(p1, p2)` and report the invariant mass to two decimal places.


In [ ]:
def four_vector(pT, eta, phi, m):
    px = pT * np.cos(phi)
    py = pT * np.sin(phi)
    pz = pT * np.sinh(eta)
    E = np.sqrt(px**2 + py**2 + pz**2 + m**2)
    return np.array([E, px, py, pz])

def inv_mass(p1, p2):
    E1, *p1_vec = p1
    E2, *p2_vec = p2
    p_sum = np.array(p1_vec) + np.array(p2_vec)
    m_squared = (E1 + E2) ** 2 - np.dot(p_sum, p_sum)
    return float(np.sqrt(max(m_squared, 0.0)))

photon_1 = four_vector(60.0, 1.2, 0.3, 0.0)
photon_2 = four_vector(70.0, -0.4, 2.5, 0.0)

m_gg = round(inv_mass(photon_1, photon_2), 2)

check("discovery.m-gg", m_gg)


### Exercise 5: find the Higgs

The cell below loads a toy dataset of 500 diphoton invariant masses, in GeV. There is a smooth continuum background from 100 – 150 GeV plus a Gaussian signal bump on top — that bump is the Higgs.

Histogram `m_gamma_gamma` into 1-GeV-wide bins from 100 to 150 GeV. Report the **integer bin center** (e.g. `124`, `125`, `126`) that contains the most events.

*Heads up:* this checkpoint is configured for **code submission** — the cell magic `%%cadence_submit` sends your code to the teacher in addition to running it. Use it on your final solution so the teacher can see your approach.


In [ ]:
%%cadence_submit discovery.higgs-peak
rng = np.random.default_rng(1)
background = rng.uniform(100.0, 150.0, size=450)
signal     = rng.normal(loc=125.0, scale=1.5, size=80)
m_gamma_gamma = np.concatenate([background, signal])

bin_edges = np.arange(100, 151)               # 100, 101, ..., 150
counts, _ = np.histogram(m_gamma_gamma, bins=bin_edges)

peak_idx = int(np.argmax(counts))
peak_bin_center = int(bin_edges[peak_idx])     # left edge == integer center

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(bin_edges[:-1], counts, width=1.0, align='edge', edgecolor='white')
ax.axvline(peak_bin_center + 0.5, color='crimson', linestyle='--', label=f'peak at {peak_bin_center} GeV')
ax.set_xlabel(r'$m_{\gamma\gamma}$  [GeV]')
ax.set_ylabel('events / GeV')
ax.set_title('Diphoton invariant mass spectrum')
ax.legend()
plt.show()

check("discovery.higgs-peak", peak_bin_center)


### Exercise 6: reflect on the result

Stuck on the peak? Once you've made a few attempts, you can ask Cadence to show the worked solution by running `cadence.show_solution("discovery.higgs-peak")`. Try it after attempt 3.

When you've found the peak, the final "checkpoint" is **manual** — there's no single right answer. In the cell below, briefly describe (in a comment or markdown cell) what the peak shape tells you about how the Higgs was discovered. Then run `mark_done("discovery.reflect")` to count yourself as finished.


In [ ]:
# Your reflection here — for example:
#   The bump at 125 GeV stands out against an otherwise smooth background. ATLAS and CMS
#   independently saw a similar excess in 2012, which is why the Higgs discovery required
#   two experiments seeing the same signal in the same place.

mark_done("discovery.reflect")


🎉 If your peak landed at 125 GeV, congratulations — you just rediscovered the Higgs boson in toy data.

Real ATLAS/CMS analyses use the same idea: build photon four-vectors, compute the invariant mass of every pair, and look for a bump. The teacher dashboard for this lesson will now show your attempts alongside everyone else's — solve rates, common wrong answers, timing, and any code you submitted via `%%cadence_submit`.
